In [1]:
import numpy as np
import pandas as pd
from unicodedata import normalize
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset,DataLoader
import torch
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter

In [2]:
# Asegurarse de correr el script de limpieza para poder ejecutar el script a continuación
%run limpieza_data.ipynb

Todos los archivos han sido cargados
dataframe expextativas limpio
dataframe tasa_politica limpio
dataframe indice_precios limpio
dataframe tasa_ibr limpio
dataframe tasa_mercado limpio
Filtro temporal aplicado
Valores del mes seleccioandos
DATAFRAME CONSOLIDADO
Columna fecha eliminada de df_modelo_sin_fecha


## Normalización de datos

In [3]:
# Configuración de parametros para la clase BanrepDataset
n = len(df_modelo_sin_fecha)
train_obs = int(n * 0.70)
val_obs = int(n * 0.85)
test_obs = n - train_obs - val_obs

scaler = StandardScaler()
scaler.fit(df_modelo_sin_fecha[:train_obs])
df_modelo_sin_fecha = scaler.transform(df_modelo_sin_fecha)

ipc_mean = scaler.mean_[2]
ipc_std = scaler.scale_[2]

In [4]:
pasos_dato = 1
sequence_length = contexto = 12
retraso = pasos_dato * (contexto + 1 - 1)
batch_size = 32

class BanrepDataset(Dataset):
    def __init__(self, data, sequence_length, target_col, indice_inicio, indice_final, sampling_rate):
        self.data = data
        self.sequence_length = sequence_length
        self.indices = np.arange(indice_inicio, indice_final)
        self.target = target_col
        self.sampling_rate = sampling_rate

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        inicio = self.indices[idx]
        pasos = np.arange(inicio, inicio + self.sequence_length * self.sampling_rate)

        x = self.data[pasos]
        y = self.data[inicio + retraso, self.target]

        return torch.tensor(x, dtype = torch.float32), torch.tensor(y, dtype = torch.float32)


In [5]:
train_dataset = BanrepDataset(
    data = df_modelo_sin_fecha,
    sequence_length = contexto, 
    target_col = 2, 
    indice_inicio = 0, indice_final = train_obs,
    sampling_rate = pasos_dato
)

val_dataset = BanrepDataset(
    data = df_modelo_sin_fecha,
    sequence_length = contexto,
    target_col = 2,
    indice_inicio = train_obs - retraso, 
    indice_final = val_obs - retraso, 
    sampling_rate = pasos_dato
)

test_dataset = BanrepDataset(
    data = df_modelo_sin_fecha,
    sequence_length = contexto,
    target_col = 2,
    indice_inicio = val_obs - retraso,
    indice_final = n - retraso,
    sampling_rate = pasos_dato
)

train_dataloader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
val_dataloader = DataLoader(val_dataset, batch_size = batch_size, shuffle = False)
test_dataloader = DataLoader(test_dataset, batch_size = batch_size, shuffle = False)

for inputs, targets in train_dataloader:
    print('Dimensiones de la entrada: ', inputs.shape)
    print('Dimensiones del objetivo: ', targets.shape)
    break


Dimensiones de la entrada:  torch.Size([32, 12, 5])
Dimensiones del objetivo:  torch.Size([32])


In [6]:
def correr_etapa(modelo, carga, criterio, optimizador = None):
    entrenamiento = optimizador is not None
    modelo.train() if entrenamiento else modelo.eval()
    perdida_total = 0.0
    mae_total = 0.0
    n = 0

    with torch.set_grad_enabled(entrenamiento):
        for entrada, objetivo in carga:
            entrada, objetivo = entrada.to(dispositivo), objetivo.to(dispositivo)
            predicc = modelo(entrada)
            perdida = criterio(predicc, objetivo)

            if entrenamiento:
                optimizador.zero_grad()
                perdida.backward()
                optimizador.step()

            perdida_total += perdida.item() * entrada.size(0)
            mae_total += torch.sum(torch.abs(predicc - objetivo)).item()
            n += entrada.size(0)

    return perdida_total / n, mae_total / n 


def prediccion(modelo, dataset): 
    modelo.eval()
    predicciones = []
    objetivos = []
    carga = DataLoader(dataset, batch_size = 32, shuffle = False)

    with torch.no_grad():
        for entrada, objetivo in carga:
            predicc = modelo(entrada.to(dispositivo)).cpu().numpy()
            predicciones.append(predicc * ipc_std + ipc_mean) 
            objetivos.append(objetivo.numpy())
    
    return np.concatenate(predicciones), np.concatenate(objetivos)

In [7]:
class punto_guardado_modelo():
    def __init__(self, directorio, vigilante = 'mae_val', comparacion = 'min', mostrar = True):
        self.filepath = directorio
        self.monitor = vigilante
        self.verbose = mostrar
        self.best = float('inf') if comparacion == 'min' else float('-inf')
        self.mode = comparacion

    def paso(self, metrica, modelo = None): 
        valor = metrica[self.monitor]
        mejora = valor < self.best if self.mode == 'min' else value > self.best
        
        if mejora:
            self.best = valor
            torch.save(modelo.state_dict(), self.filepath)
        
        if self.verbose:
            print(f'Mejor modelo guardado {self.monitor}: {valor:.2f} Puntos')

In [8]:
class parada_temprana():
    def __init__(self, vigilante = 'mae_val', paciencia = 5, min_delta = 1e-4, comparacion = 'min'):
        self.vigilante = vigilante
        self.paciencia = paciencia
        self.val_min = min_delta
        self.comparar = comparacion
        self.mejor = float('inf') if comparacion == 'min' else  float('-inf')
        self.contador = 0
        self.parar = False

        def paso(self, metrica, model = None):
            valor = metrica[self.vigilante]
            mejora = (valor < self.best if self.comparacion == 'min' else valor > self.mejor + self.val_min)

            if mejora: 
                self.mejor = valor
                self.contador = 0
            else:
                self.contador += 1 
                if self.contador > self.paciencia:
                    self.parar = True
                    print(f'Se activa parada temprana debido a no mejoramiento del MAE en {self.paciencia} Etapas')
            return mejora      

In [9]:
class reducir_LR:
    def __init__(self, optimizador, vigilante = "mae_val", paciencia = 3, factor = 0.5):
        self.lrprogramador = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizador, patience = paciencia, factor = factor
        )
        self.vigilante = vigilante
    
    def paso(self, metrica, model = None):
        self.lrprogramador.paso(metrica[self.vigilante])

In [10]:
class rnn_simple(nn.Module):
    def __init__(self, caracteristicas, neuronas = 3):
        super().__init__()
        self.rnn = nn.RNN(input_size = caracteristicas, hidden_size = neuronas, batch_first = True)
        self.head = nn.Linear(neuronas, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1, :]).squeeze(-1)

In [11]:
dispositivo = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
modelo = rnn_simple(caracteristicas = df_modelo_sin_fecha.shape[-1]).to(dispositivo)
optimizador = torch.optim.Adam(modelo.parameters())
criterio = nn.MSELoss()
writer = SummaryWriter(log_dir = 'runs/modelo_RNN')

llamadas = [
    punto_guardado_modelo('Mejor_modelo_simpleRNN.pt', vigilante = 'mae_val')
]

epocas = 10

for epoca in range(epocas + 1):
    perd_entreno, mae_entreno = correr_etapa(modelo, train_dataloader, criterio, optimizador)
    perd_val, mae_val = correr_etapa(modelo, val_dataloader, criterio)

    metricas = {'perd_entreno': perd_entreno, 'mae_entreno': mae_entreno, 
                'perd_val': perd_val, 'mae_val': mae_val}
    
    writer.add_scalars('Perdida', {'Entrenamiento': perd_entreno, 'Validacion': perd_val}, epoca)
    writer.add_scalars('MAE', {'Entrenamiento': mae_entreno, 'Validacion': mae_val}, epoca)

    print(f'Epoca - {epoca:02d} - '
          f'Perdida en entrenamiento: {perd_entreno:.4f}, MAE en entrenamiento: {mae_entreno:.04f} |'
          f'Perdida en validación: {perd_val:.4f}, MAE en validación: {mae_val:.4f}')

    for llamada in llamadas:
        llamada.paso(metricas, modelo) if isinstance(llamada, punto_guardado_modelo) else llamada.paso(metricas)

writer.close()


Epoca - 00 - Perdida en entrenamiento: 0.7088, MAE en entrenamiento: 0.7505 |Perdida en validación: 5.0458, MAE en validación: 2.1091
Mejor modelo guardado mae_val: 2.11 Puntos
Epoca - 01 - Perdida en entrenamiento: 0.6928, MAE en entrenamiento: 0.7423 |Perdida en validación: 5.0042, MAE en validación: 2.0957
Mejor modelo guardado mae_val: 2.10 Puntos
Epoca - 02 - Perdida en entrenamiento: 0.6764, MAE en entrenamiento: 0.7339 |Perdida en validación: 4.9646, MAE en validación: 2.0834
Mejor modelo guardado mae_val: 2.08 Puntos
Epoca - 03 - Perdida en entrenamiento: 0.6609, MAE en entrenamiento: 0.7255 |Perdida en validación: 4.9242, MAE en validación: 2.0705
Mejor modelo guardado mae_val: 2.07 Puntos
Epoca - 04 - Perdida en entrenamiento: 0.6458, MAE en entrenamiento: 0.7175 |Perdida en validación: 4.8810, MAE en validación: 2.0567
Mejor modelo guardado mae_val: 2.06 Puntos
Epoca - 05 - Perdida en entrenamiento: 0.6302, MAE en entrenamiento: 0.7093 |Perdida en validación: 4.8402, MAE en 

In [12]:
modelo.load_state_dict(torch.load('Mejor_modelo_simpleRNN.pt', map_location = dispositivo))
_, test_mae = correr_etapa(modelo, test_dataloader, criterio)
print(f'Test MAE: {test_mae:.2f}')


Test MAE: 4.08


In [13]:
class rnn_multi_capas(nn.Module):
    def __init__(self, caracteristicas, neuronas = 3, capas = 3):
        super().__init__()
        self.rnn = nn.RNN(input_size = caracteristicas, hidden_size = neuronas, num_layers = capas, batch_first = True)
        self.head = nn.Linear(neuronas, 1)

    def forward(self, x):
        x, _ = self.rnn(x)
        return self.head(x[:, -1, :]).squeeze(-1)


In [14]:
modelo = rnn_multi_capas(caracteristicas = df_modelo_sin_fecha.shape[-1]).to(dispositivo)
writer = SummaryWriter(log_dir = 'runs/Modelo_RNN_2_capas.pt')
optimizador = torch.optim.Adam(modelo.parameters())
criterio = nn.MSELoss()
dispositivo = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

llamadas = [
    punto_guardado_modelo('Mejor_modelo_RNN_multi_capas.pt', vigilante = 'mae_val')
]

for epoca in range(epocas + 1):
    perd_entreno, mae_entreno = correr_etapa(modelo, train_dataloader, criterio, optimizador)
    perd_val, mae_val = correr_etapa(modelo, val_dataloader, criterio)

    metricas = {'perd_entreno': perd_entreno, 'mae_entreno': mae_entreno, 
                'perd_val': perd_val, 'mae_val': mae_val}
    
    writer.add_scalars('Perdida', {'Entrenamiento': perd_entreno, 'Validacion': perd_val}, epoca)
    writer.add_scalars('MAE', {'Entrenamiento': mae_entreno, 'Validacion': mae_val}, epoca)

    print(f'Epoca - {epoca:02d} - '
          f'Perdida en entrenamiento: {perd_entreno:.4f}, MAE en entrenamiento: {mae_entreno:.04f} |'
          f'Perdida en validación: {perd_val:.4f}, MAE en validación: {mae_val:.4f}')

    for llamada in llamadas:
        llamada.paso(metricas, modelo) if isinstance(llamada, punto_guardado_modelo) else llamada.paso(metricas)

writer.close()


Epoca - 00 - Perdida en entrenamiento: 0.9883, MAE en entrenamiento: 0.9033 |Perdida en validación: 4.9911, MAE en validación: 2.0934
Mejor modelo guardado mae_val: 2.09 Puntos
Epoca - 01 - Perdida en entrenamiento: 0.9488, MAE en entrenamiento: 0.8861 |Perdida en validación: 5.0823, MAE en validación: 2.1152
Mejor modelo guardado mae_val: 2.12 Puntos
Epoca - 02 - Perdida en entrenamiento: 0.9115, MAE en entrenamiento: 0.8684 |Perdida en validación: 5.1607, MAE en validación: 2.1338
Mejor modelo guardado mae_val: 2.13 Puntos
Epoca - 03 - Perdida en entrenamiento: 0.8759, MAE en entrenamiento: 0.8487 |Perdida en validación: 5.2107, MAE en validación: 2.1453
Mejor modelo guardado mae_val: 2.15 Puntos
Epoca - 04 - Perdida en entrenamiento: 0.8433, MAE en entrenamiento: 0.8299 |Perdida en validación: 5.2407, MAE en validación: 2.1517
Mejor modelo guardado mae_val: 2.15 Puntos
Epoca - 05 - Perdida en entrenamiento: 0.8141, MAE en entrenamiento: 0.8118 |Perdida en validación: 5.2562, MAE en 

In [15]:
modelo.load_state_dict(torch.load('Mejor_modelo_RNN_multi_capas.pt', map_location = dispositivo))

_, test_mae = correr_etapa(modelo, test_dataloader, criterio)
print(f'Test MAE: {test_mae:.4f}')

Test MAE: 4.4318
